In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .appName("codespaces-spark") \
    .config("spark.ui.port", "4050") \
    .config("spark.ui.enabled", "true") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.local.dir", "/tmp") \
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse") \
    .getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/04 09:06:36 WARN Utils: Your hostname, codespaces-b285a2, resolves to a loopback address: 127.0.0.1; using 10.0.0.177 instead (on interface eth0)
26/03/04 09:06:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 09:06:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/04 09:06:37 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


Spark version: 4.1.1


In [7]:
!wget -O /tmp/fhvhv_tripdata_2021-01.csv.gz https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-04 09:12:00--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 20.207.73.82
Connecting to github.com (github.com)|20.207.73.82|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-04T10%3A10%3A32Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-04T09%3A10%3A12Z&ske=2026-03-04T10%3A10%3A32Z&sks=b&skv=2018-11-09&sig=WFQCKe7svLDxcwpva7QiOB9lbBt%2FM4oBYEC36HJy%2FQQ%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjYxOTEyMCwibmJmIjoxNzcyNjE1NTIwLCJwYXRo

In [8]:
!ls -lh /tmp/fhvhv_tripdata_2021-01.csv.gz

-rw-r--rw- 1 codespace codespace 124M Jul 14  2022 /tmp/fhvhv_tripdata_2021-01.csv.gz


In [37]:
!gzip -dc /tmp/fhvhv_tripdata_2021-01.csv.gz > /tmp/fhvhv_tripdata_2021-01.csv

In [38]:
!wc -l /tmp/fhvhv_tripdata_2021-01.csv

11908469 /tmp/fhvhv_tripdata_2021-01.csv


In [39]:
df = spark.read \
    .option("header", "true") \
    .csv("/tmp/fhvhv_tripdata_2021-01.csv")

In [40]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [41]:
!head -n 1001 /tmp/fhvhv_tripdata_2021-01.csv > /tmp/head.csv

In [42]:
import pandas as pd

In [43]:
df_pandas = pd.read_csv("/tmp/head.csv")

In [44]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [45]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [46]:
from pyspark.sql import types

In [47]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [49]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('/tmp/fhvhv_tripdata_2021-01.csv')

In [50]:
df = df.repartition(24)

In [60]:
df.write.parquet('/tmp/fhvhv/2021/01/', mode='overwrite')

In [61]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [63]:
!ls -lh /tmp/fhvhv/2021/01/

total 215M
-rw-r--r-- 1 codespace codespace    0 Mar  4 10:02 _SUCCESS
-rw-r--r-- 1 codespace codespace 9.0M Mar  4 10:02 part-00000-90edc710-6a50-4dc4-a171-02dab2597deb-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  4 10:02 part-00001-90edc710-6a50-4dc4-a171-02dab2597deb-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  4 10:02 part-00002-90edc710-6a50-4dc4-a171-02dab2597deb-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  4 10:02 part-00003-90edc710-6a50-4dc4-a171-02dab2597deb-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  4 10:02 part-00004-90edc710-6a50-4dc4-a171-02dab2597deb-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  4 10:02 part-00005-90edc710-6a50-4dc4-a171-02dab2597deb-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  4 10:02 part-00006-90edc710-6a50-4dc4-a171-02dab2597deb-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 9.0M Mar  4 10:02 part-00007-90edc710-6a50-4dc4-a171-02dab2597d

In [64]:
#Cleanup cell AFTER jobs
import shutil, os

def clear_tmp():
    tmp_path = "/tmp/spark-warehouse"
    if os.path.exists(tmp_path):
        shutil.rmtree(tmp_path)
    print("Cleared /tmp Spark warehouse")

clear_tmp()


Cleared /tmp Spark warehouse


In [65]:
!ls -lh /tmp

total 842M
drwxr-xrw-+  2 codespace codespace 4.0K Mar  4 09:06 artifacts-30fd8fbf-f5d2-465c-95ed-f935cff3a4b7
drwxr-xrw-+ 66 codespace codespace 4.0K Mar  4 09:52 blockmgr-3208d09e-96ad-4c9a-af80-40608eab457d
-rw-r--rw-   1 root      root       12K Mar  4 07:12 dockerd.log
drwxr-xr-x+  3 codespace codespace 4.0K Mar  4 09:52 fhvhv
-rw-r--rw-   1 codespace codespace 718M Mar  4 09:44 fhvhv_tripdata_2021-01.csv
-rw-r--rw-   1 codespace codespace 124M Jul 14  2022 fhvhv_tripdata_2021-01.csv.gz
-rw-r--rw-   1 codespace codespace  62K Mar  4 09:45 head.csv
drwxr-xr--+  2 codespace codespace 4.0K Mar  4 09:06 hsperfdata_codespace
-rw-r--rw-   1 codespace codespace 152K Mar  4 09:12 liblz4-java-8498321997344148136.so
-rw-r--rw-   1 codespace codespace    0 Mar  4 09:12 liblz4-java-8498321997344148136.so.lck
drwx------+  2 codespace codespace 4.0K Mar  4 08:28 mcp-5CWuZP
drwx------+  2 codespace codespace 4.0K Mar  4 07:34 mcp-GqWa9J
drwx------+  2 codespace codespace 4.0K Mar  4 07:27 mcp-Kl